In [1]:
%reset -f

In [2]:
import scanpy as sc
import pandas as pd
import scipy.io
from scipy.sparse import csr_matrix

In [3]:
import scipy.io
from scipy.sparse import csr_matrix
import anndata as ad
import numpy as np

In [4]:
import os

In [5]:
# Set correct path
base_path = r"C:\Users\annam\Downloads\GSE176078_Wu_etal_2021_BRCA_scRNASeq\Wu_etal_2021_BRCA_scRNASeq"

In [6]:
# Check files
print(os.listdir(base_path))

['count_matrix_barcodes.tsv', 'count_matrix_genes.tsv', 'count_matrix_sparse.mtx', 'metadata.csv']


In [8]:
# 1. Load matrix
X = scipy.io.mmread(base_path + r"\count_matrix_sparse.mtx")

# 2. Convert to CSR sparse format
X = csr_matrix(X, dtype=np.float32)

# 3. Create AnnData object
adata = ad.AnnData(X)

# 4. Load genes and barcodes
genes = pd.read_csv(
    base_path + r"\count_matrix_genes.tsv",
    header=None,
    sep="\t"
)

barcodes = pd.read_csv(
    base_path + r"\count_matrix_barcodes.tsv",
    header=None,
    sep="\t"
)

# 5. Since matrix is genes x cells
adata.obs_names = genes[0].astype(str).values
adata.var_names = barcodes[0].astype(str).values

# 6. Transpose to cells x genes
adata = adata.T

# 7. Make names unique
adata.var_names_make_unique()
adata.obs_names_make_unique()

print(adata)
print(adata.obs_names[:5])
print(adata.var_names[:5])

AnnData object with n_obs × n_vars = 100064 × 29733
Index(['CID3586_AAGACCTCAGCATGAG', 'CID3586_AAGGTTCGTAGTACCT',
       'CID3586_ACCAGTAGTTGTGGCC', 'CID3586_ACCCACTAGATGTCGG',
       'CID3586_ACTGATGGTCAACTGT'],
      dtype='object')
Index(['RP11-34P13.7', 'FO538757.3', 'FO538757.2', 'AP006222.2',
       'RP4-669L17.10'],
      dtype='object')


In [9]:
metadata = pd.read_csv(base_path + r"\metadata.csv")

metadata.index = metadata["Unnamed: 0"].astype(str)

adata.obs = metadata.loc[adata.obs_names].copy()

adata.obs["dataset"] = "GSE176078"

print(adata)
print(adata.obs.head())

AnnData object with n_obs × n_vars = 100064 × 29733
    obs: 'Unnamed: 0', 'orig.ident', 'nCount_RNA', 'nFeature_RNA', 'percent.mito', 'subtype', 'celltype_subset', 'celltype_minor', 'celltype_major', 'dataset'
                                        Unnamed: 0 orig.ident  nCount_RNA  \
CID3586_AAGACCTCAGCATGAG  CID3586_AAGACCTCAGCATGAG    CID3586        4581   
CID3586_AAGGTTCGTAGTACCT  CID3586_AAGGTTCGTAGTACCT    CID3586        1726   
CID3586_ACCAGTAGTTGTGGCC  CID3586_ACCAGTAGTTGTGGCC    CID3586        1229   
CID3586_ACCCACTAGATGTCGG  CID3586_ACCCACTAGATGTCGG    CID3586        1352   
CID3586_ACTGATGGTCAACTGT  CID3586_ACTGATGGTCAACTGT    CID3586        1711   

                          nFeature_RNA  percent.mito subtype  \
CID3586_AAGACCTCAGCATGAG          1689      1.506221   HER2+   
CID3586_AAGGTTCGTAGTACCT           779      5.793743   HER2+   
CID3586_ACCAGTAGTTGTGGCC           514      1.383238   HER2+   
CID3586_ACCCACTAGATGTCGG           609      1.923077   HER2+   
CID358

In [14]:
from pathlib import Path

output_dir = Path(r"C:\Users\annam\Dissertation 2026\Data\Raw")
output_dir.mkdir(parents=True, exist_ok=True)

adata.write(output_dir / "GSE176078_raw.h5ad")

print("Saved successfully")

Saved successfully


In [15]:
adata2 = sc.read_h5ad(
    r"C:\Users\annam\Dissertation 2026\Data\Raw\GSE176078_raw.h5ad"
)

print(adata2)

AnnData object with n_obs × n_vars = 100064 × 29733
    obs: 'Unnamed: 0', 'orig.ident', 'nCount_RNA', 'nFeature_RNA', 'percent.mito', 'subtype', 'celltype_subset', 'celltype_minor', 'celltype_major', 'dataset'


In [12]:
print(adata2)
print(adata2.obs.shape)
print(adata2.var.shape)
print(adata2.obs.columns)
print(adata2.obs.head())
print(adata2.X.shape)

AnnData object with n_obs × n_vars = 100064 × 29733
    obs: 'Unnamed: 0', 'orig.ident', 'nCount_RNA', 'nFeature_RNA', 'percent.mito', 'subtype', 'celltype_subset', 'celltype_minor', 'celltype_major', 'dataset'
(100064, 10)
(29733, 0)
Index(['Unnamed: 0', 'orig.ident', 'nCount_RNA', 'nFeature_RNA',
       'percent.mito', 'subtype', 'celltype_subset', 'celltype_minor',
       'celltype_major', 'dataset'],
      dtype='object')
                                        Unnamed: 0 orig.ident  nCount_RNA  \
CID3586_AAGACCTCAGCATGAG  CID3586_AAGACCTCAGCATGAG    CID3586        4581   
CID3586_AAGGTTCGTAGTACCT  CID3586_AAGGTTCGTAGTACCT    CID3586        1726   
CID3586_ACCAGTAGTTGTGGCC  CID3586_ACCAGTAGTTGTGGCC    CID3586        1229   
CID3586_ACCCACTAGATGTCGG  CID3586_ACCCACTAGATGTCGG    CID3586        1352   
CID3586_ACTGATGGTCAACTGT  CID3586_ACTGATGGTCAACTGT    CID3586        1711   

                          nFeature_RNA  percent.mito subtype  \
CID3586_AAGACCTCAGCATGAG          1689     